# Setup



install dependencies

In [1]:
# PyMuPDF: PDF text extraction (same as assignment 3)
# google-genai: official Google SDK for the Gemini API (embeddings)
# chromadb: local vector database
!pip install -q pymupdf google-genai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

Configure Gemini API key and smoke-test the embedding model

In [2]:
import numpy as np
from google import genai
from google.colab import userdata

# Read the API key from Colab Secrets (never hard-code it!)
api_key = userdata.get("GOOGLE_API_KEY")

# Create the Gemini client
client = genai.Client(api_key=api_key)

# --- Smoke test: embed three words and compare their similarities ---
words = ["dog", "cat", "car"]
result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=words,
)

# result.embeddings is a list; .values holds the actual float vector
vectors = [np.array(e.values) for e in result.embeddings]
print(f"Embedding dimension: {len(vectors[0])}")

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity = dot product of the two normalized vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"dog vs cat: {cosine_similarity(vectors[0], vectors[1]):.4f}")
print(f"cat vs car: {cosine_similarity(vectors[1], vectors[2]):.4f}")
print(f"dog vs car: {cosine_similarity(vectors[0], vectors[2]):.4f}")

Embedding dimension: 3072
dog vs cat: 0.7469
cat vs car: 0.6562
dog vs car: 0.6309


# Load and chunk document

upload a PDF and extract its text

In [3]:
import pymupdf
from google.colab import files

# Upload a PDF from your knowledge base
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

def load_pdf(path: str) -> str:
    """Read a PDF file and return all of its text as one string.

    `sort=True` orders text top-left -> bottom-right, which helps with
    the multi-column layouts common in research papers.
    """
    doc = pymupdf.open(path)
    pages_text = [page.get_text(sort=True) for page in doc]
    doc.close()
    return "\n".join(pages_text)

document_text = load_pdf(pdf_filename)
print(f"Loaded: {pdf_filename}")
print(f"Total characters extracted: {len(document_text)}")

Saving ColBERT.pdf to ColBERT.pdf
Loaded: ColBERT.pdf
Total characters extracted: 83038


fixed-size chunks with overlap (from assignment 3)

In [4]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 100) -> list[str]:
    """Split `text` into fixed-size chunks that overlap each other.

    Consecutive chunks share `overlap` characters, so a sentence sitting
    on a boundary still appears whole in at least one chunk.
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    step = chunk_size - overlap
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += step
    return chunks

CHUNK_SIZE = 800
OVERLAP = 100

chunks = chunk_text(document_text, chunk_size=CHUNK_SIZE, overlap=OVERLAP)

# Metadata for each chunk: which file it came from and its position.
# (Later versions will add the page number here.)
metadatas = [{"source": pdf_filename, "chunk_index": i} for i in range(len(chunks))]

print(f"Number of chunks: {len(chunks)}")
print("\n----- Chunk 0 preview -----")
print(chunks[0][:300])

Number of chunks: 119

----- Chunk 0 preview -----
          ColBERT: Eﬀicient and Eﬀective Passage Search via
               Contextualized Late Interaction over BERT

                    Omar Khatab                                Matei Zaharia
                               Stanford University                                      Stanford Universi


# Embedding functions

batched document embedding + query embedding

In [5]:
import time
from google.genai import types

EMBEDDING_MODEL = "gemini-embedding-001"
BATCH_SIZE = 20          # texts per API request
SLEEP_BETWEEN_BATCHES = 2  # seconds; stay well under free-tier rate limits


def _embed(texts: list[str], task_type: str) -> list[list[float]]:
    """Embed a list of texts in batches. Retries once on rate-limit errors."""
    all_vectors = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        try:
            result = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=batch,
                config=types.EmbedContentConfig(task_type=task_type),
            )
        except Exception as e:
            # Simple retry: wait and try the same batch once more
            print(f"Batch {i // BATCH_SIZE} failed ({e}); retrying in 30s...")
            time.sleep(30)
            result = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=batch,
                config=types.EmbedContentConfig(task_type=task_type),
            )
        all_vectors.extend([e.values for e in result.embeddings])
        print(f"Embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)} texts")
        if i + BATCH_SIZE < len(texts):
            time.sleep(SLEEP_BETWEEN_BATCHES)
    return all_vectors


def embed_documents(texts: list[str]) -> list[list[float]]:
    """Embed knowledge-base chunks (document side of retrieval)."""
    return _embed(texts, task_type="RETRIEVAL_DOCUMENT")


def embed_query(text: str) -> list[float]:
    """Embed a user question (query side of retrieval)."""
    return _embed([text], task_type="RETRIEVAL_QUERY")[0]

Quick test with two tiny texts

In [6]:
test_vectors = embed_documents(["late interaction", "query encoder"])
print(f"\nGot {len(test_vectors)} vectors, dimension = {len(test_vectors[0])}")

Embedded 2/2 texts

Got 2 vectors, dimension = 3072


# VectorStore

a ChromaDB-backed vector database with add / query

In [7]:
import chromadb


class VectorStore:
    """A thin wrapper around ChromaDB using Gemini embeddings.

    We pass embeddings explicitly (instead of using Chroma's built-in
    embedding model) so that documents and queries are guaranteed to be
    embedded by the same model.
    """

    def __init__(self, collection_name: str = "papers", path: str = "./chroma_db"):
        self._client = chromadb.PersistentClient(path=path)
        self._collection = self._client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},  # cosine distance, not default L2
        )

    def add(self, texts: list[str], metadatas: list[dict]) -> None:
        """Embed the chunks and store (vector, text, metadata) in the DB."""
        vectors = embed_documents(texts)
        # Chroma requires a unique string ID per entry
        ids = [f"{m['source']}-{m['chunk_index']}" for m in metadatas]
        self._collection.add(
            ids=ids,
            embeddings=vectors,
            documents=texts,
            metadatas=metadatas,
        )
        print(f"Added {len(texts)} chunks. Collection size: {self.count()}")

    def query(self, question: str, top_k: int = 5) -> list[dict]:
        """Embed the question and return the top_k most similar chunks."""
        query_vector = embed_query(question)
        results = self._collection.query(
            query_embeddings=[query_vector],
            n_results=top_k,
        )
        # Re-pack Chroma's column-oriented result into a list of dicts
        return [
            {
                "text": doc,
                "metadata": meta,
                "distance": dist,  # cosine distance: smaller = more similar
            }
            for doc, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]

    def count(self) -> int:
        """Number of chunks currently stored."""
        return self._collection.count()

Build the vector database, embed and store all chunks

In [8]:
store = VectorStore(collection_name="papers")

store.add(chunks, metadatas)

Embedded 20/119 texts
Embedded 40/119 texts
Embedded 60/119 texts
Embedded 80/119 texts
Embedded 100/119 texts
Batch 5 failed (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 47.461712665s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedConte

# End-to-end retrieval test

ask real questions against the paper

In [9]:
def show_results(question: str, top_k: int = 3) -> None:
    """Pretty-print the top_k retrieved chunks for a question."""
    print("=" * 80)
    print(f"QUESTION: {question}")
    for rank, r in enumerate(store.query(question, top_k=top_k), start=1):
        meta = r["metadata"]
        print(f"\n--- Rank {rank} | distance={r['distance']:.4f} "
              f"| {meta['source']} chunk#{meta['chunk_index']} ---")
        print(r["text"][:400])
    print()


# Core-concept question
show_results("What is late interaction in ColBERT?")

# Comparison / efficiency question
show_results("How does ColBERT's computational cost compare to BERT-based rankers?")

# Detail question
show_results("Which similarity operator does ColBERT use to score query and document embeddings?")

# Negative control: a topic that does NOT exist in the paper.
# Expect clearly larger distances than the questions above.
show_results("What is the best recipe for chocolate cake?")

QUESTION: What is late interaction in ColBERT?
Embedded 1/1 texts

--- Rank 1 | distance=0.2466 | ColBERT.pdf chunk#5 ---
RT introduces a late interaction architecture that indepen-Jun
           dently encodes the query and the document using BERT and then      Figure 1: Eﬀectiveness (MRR@10) versus Mean Qery La-
4    employs a cheap yet powerful interaction step that models their     tency (log-scale) for a number of representative ranking
          ﬁne-grained similarity. By delaying and yet retaining this ﬁne-   

--- Rank 2 | distance=0.2470 | ColBERT.pdf chunk#20 ---
timating relevance between a query      top-k results directly from a large document collection, substan-
q and a document d. Under late interaction, q and d are separately        tially improving recall over models that only re-rank the output of
encoded into two sets of contextual embeddings, and relevance is      term-based retrieval.
evaluated using cheap and pruning-friendly computations betw

--- Rank 3 | dista